# Claude Sonnet 4.6 本地集成 / Claude Sonnet 4.6 Local Integration

本笔记展示如何在本地使用 Claude Sonnet 4.6（`claude-sonnet-4-5`）模型，通过 Anthropic Python SDK 与模型交互。

This notebook demonstrates how to use the Claude Sonnet 4.6 model locally via the Anthropic Python SDK.

## 1. 安装依赖 / Install Dependencies

In [ ]:
# 安装 Anthropic Python SDK
!pip install anthropic -q

## 2. 配置 API Key / Configure API Key

请将您的 Anthropic API Key 设置为环境变量 `ANTHROPIC_API_KEY`，或直接在下方代码中填入。

Set your Anthropic API Key as the environment variable `ANTHROPIC_API_KEY`, or enter it directly below.

In [ ]:
import os

# 方式一：从环境变量读取（推荐）
# Option 1: Read from environment variable (recommended)
api_key = os.environ.get("ANTHROPIC_API_KEY", "")

# 方式二：直接填写（不推荐提交到版本控制）
# Option 2: Enter directly (not recommended to commit to version control)
# api_key = "sk-ant-..."

if not api_key:
    print("⚠️  请设置 ANTHROPIC_API_KEY 环境变量或在上方直接填写 API Key")
    print("⚠️  Please set the ANTHROPIC_API_KEY environment variable or enter the API Key above")
else:
    print("✅ API Key 已加载 / API Key loaded")

## 3. 初始化客户端 / Initialize Client

In [ ]:
import anthropic

# 初始化 Anthropic 客户端
# Initialize the Anthropic client
client = anthropic.Anthropic(api_key=api_key)

# Claude Sonnet 4.6 的模型 ID
# Model ID for Claude Sonnet 4.6
MODEL_NAME = "claude-sonnet-4-5"

print(f"使用模型 / Using model: {MODEL_NAME}")

## 4. 基本文本生成 / Basic Text Generation

In [ ]:
def chat(prompt, max_tokens=1024):
    """向 Claude Sonnet 4.6 发送消息并获取回复
    
    Send a message to Claude Sonnet 4.6 and get a response.
    """
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=max_tokens,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text


# 示例：简单问答 / Example: simple Q&A
response = chat("你好！请用中文简单介绍一下你自己。")
print(response)

## 5. 多轮对话 / Multi-turn Conversation

In [ ]:
def multi_turn_chat(conversation_history, user_message, max_tokens=1024):
    """支持多轮对话的函数
    
    Function supporting multi-turn conversations.
    """
    conversation_history.append({"role": "user", "content": user_message})
    
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=max_tokens,
        messages=conversation_history
    )
    
    assistant_reply = message.content[0].text
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply, conversation_history


# 示例多轮对话 / Example multi-turn conversation
history = []

reply1, history = multi_turn_chat(history, "请解释什么是全波形反演（FWI）？")
print("用户: 请解释什么是全波形反演（FWI）？")
print(f"Claude: {reply1}\n")

reply2, history = multi_turn_chat(history, "它在地震勘探中有哪些应用？")
print("用户: 它在地震勘探中有哪些应用？")
print(f"Claude: {reply2}")

## 6. 流式输出 / Streaming Output

In [ ]:
def stream_chat(prompt, max_tokens=1024):
    """流式输出模式，适合长文本生成
    
    Streaming output mode, suitable for long text generation.
    """
    print("Claude（流式输出）: ", end="", flush=True)
    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    print()  # 换行


# 示例流式输出 / Example streaming output
stream_chat("请用三句话描述深度学习在地球物理领域的最新进展。")

## 7. 使用 System Prompt / Using System Prompt

In [ ]:
def chat_with_system(system_prompt, user_message, max_tokens=1024):
    """使用系统提示词引导模型行为
    
    Use a system prompt to guide model behavior.
    """
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_message}
        ]
    )
    return message.content[0].text


# 示例：专业地球物理学家角色 / Example: expert geophysicist role
system = "你是一位专业的地球物理学家，擅长地震波形反演和深度学习方法。请用简洁专业的语言回答问题。"
user_msg = "CAFormer 模型相比传统 CNN 在 FWI 任务中有何优势？"

response = chat_with_system(system, user_msg)
print(f"用户: {user_msg}")
print(f"Claude（地球物理专家角色）: {response}")

## 8. 查看模型信息 / Model Information

In [ ]:
# 打印使用的模型信息
print("=" * 50)
print("模型配置 / Model Configuration")
print("=" * 50)
print(f"模型名称 / Model Name : {MODEL_NAME}")
print(f"提供商 / Provider    : Anthropic")
print(f"SDK 版本 / SDK Version: {anthropic.__version__}")
print("=" * 50)